# SLM calibration analysis

Run this notebook after `calibration_scan.py`. Configure the ROI and selected brightness branch below, then run all cells in order.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from scipy.interpolate import PchipInterpolator
from scipy.ndimage import gaussian_filter1d
from scipy.signal import savgol_filter


In [ ]:
# Configuration
def locate_calibration_dir():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        direct = candidate if candidate.name == 'calibration' else None
        nested = candidate / 'scripts' / 'calibration'
        if direct is not None and (direct / 'calibration.ipynb').is_file():
            return direct
        if nested.is_dir():
            return nested
    raise FileNotFoundError('Could not locate scripts/calibration from the notebook working directory.')

CALIBRATION_DIR = locate_calibration_dir()
INPUT_PATH = CALIBRATION_DIR / 'results' / 'current'
OUTPUT_PATH = CALIBRATION_DIR / 'results' / 'analysis'
REFERENCE_LUT_PATH = CALIBRATION_DIR / 'reference' / 'slm3324_at635_DVI.lut'
GENERATED_LUT_PATH = OUTPUT_PATH / 'custom_slm_lut.lut'

FRAME_INDEX = 0
X1, X2 = 470, 680
Y = 450
BRIGHTNESS_MIN = 47999
BRIGHTNESS_MAX = 63000
SMOOTH_PHASE = True
SAVGOL_WINDOW_LENGTH = 51
SAVGOL_POLYORDER = 3


In [ ]:
def capture_value(path):
    match = re.search(r'(\d+)\.tiff?$', path.name, flags=re.IGNORECASE)
    if match is None:
        raise ValueError(f'Capture filename does not end with a numeric value: {path.name}')
    return int(match.group(1))

capture_files = sorted([*INPUT_PATH.glob('*.tif'), *INPUT_PATH.glob('*.tiff')])
if not capture_files:
    raise FileNotFoundError(f'No TIFF captures found in {INPUT_PATH}. Run calibration_scan.py or update INPUT_PATH.')

capture_paths = {}
for path in capture_files:
    value = capture_value(path)
    if value in capture_paths:
        raise ValueError(f'Multiple captures found for mirror value {value}: {capture_paths[value].name}, {path.name}')
    capture_paths[value] = path

if FRAME_INDEX not in capture_paths:
    raise FileNotFoundError(f'No capture found for FRAME_INDEX={FRAME_INDEX:05d} in {INPUT_PATH}')

preview_path = capture_paths[FRAME_INDEX]
image = np.array(Image.open(preview_path))
x_start, x_end = sorted((X1, X2))
if not (0 <= Y < image.shape[0] and 0 <= x_start <= x_end < image.shape[1]):
    raise ValueError(f'ROI (x={x_start}:{x_end}, y={Y}) is outside image shape {image.shape}')

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
intensity_profile = image[Y, x_start:x_end + 1]
lo, hi = np.percentile(image, [1, 99.8])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
ax1.imshow(image, cmap='gray', vmin=lo, vmax=hi)
ax1.plot([X1, X2], [Y, Y], color='red', linewidth=2)
ax1.set_title(f'Image view: {preview_path.name}')
ax1.set_xlim(0, image.shape[1])
ax1.set_ylim(image.shape[0], 0)
ax2.plot(np.arange(x_start, x_end + 1), intensity_profile, color='blue', linewidth=2)
ax2.set(xlabel='X pixel', ylabel='Intensity', title='Intensity along selected line')
ax2.grid(True)
plt.tight_layout()
fig.savefig(OUTPUT_PATH / 'roi_preview.png', dpi=150)
plt.show()


In [ ]:
profiles_dict = {}
for mirror_value, path in sorted(capture_paths.items()):
    img = np.array(Image.open(path))
    if img.shape != image.shape:
        raise ValueError(f'Capture {path.name} has shape {img.shape}; expected {image.shape}')
    profiles_dict[mirror_value] = img[Y, x_start:x_end + 1]

print(f'Loaded {len(profiles_dict)} intensity profiles from {INPUT_PATH}')


In [ ]:
def fringe_phase_fft(profile, carrier_index=None, band_width=3):
    signal = profile.astype(float)
    signal = signal - gaussian_filter1d(signal, sigma=15)
    spectrum = np.fft.fft(signal)
    magnitude = np.abs(spectrum)
    magnitude[0] = 0

    if carrier_index is None:
        carrier_index = np.argmax(magnitude[1:len(signal) // 2]) + 1

    filtered = np.zeros_like(spectrum, dtype=complex)
    lower = max(1, carrier_index - band_width)
    upper = min(len(spectrum), carrier_index + band_width + 1)
    filtered[lower:upper] = spectrum[lower:upper]
    return np.angle(np.fft.ifft(filtered)), carrier_index

reference_value = min(profiles_dict)
reference_phase, carrier_index = fringe_phase_fft(profiles_dict[reference_value])
phase_shifts = {}
for mirror_value, profile in sorted(profiles_dict.items()):
    phase, _ = fringe_phase_fft(profile, carrier_index=carrier_index)
    phase_shifts[mirror_value] = np.mean(np.unwrap(phase - reference_phase))

phase_shift_figure = plt.figure(figsize=(8, 6))
plt.scatter(list(phase_shifts), list(phase_shifts.values()), color='blue', s=8)
plt.xlabel('Mirror value')
plt.ylabel('Average phase shift (radians)')
plt.title('Phase shift vs mirror value')
plt.grid(True)
phase_shift_figure.savefig(OUTPUT_PATH / 'phase_shift_vs_mirror_value.png', dpi=150)
plt.show()


In [ ]:
brightness = np.array(sorted(phase_shifts))
phase = np.array([phase_shifts[value] for value in brightness])
branch = (brightness >= BRIGHTNESS_MIN) & (brightness <= BRIGHTNESS_MAX)
brightness = brightness[branch]
phase = phase[branch]
if len(brightness) < 2:
    raise ValueError('The selected brightness range has fewer than two measured points.')

phase = np.unwrap(phase)
phase = phase - phase[0]
if phase[-1] < phase[0]:
    phase = -phase
    phase = phase - phase[0]

phase_smooth = phase.copy()
if SMOOTH_PHASE:
    window_length = min(SAVGOL_WINDOW_LENGTH, len(phase) if len(phase) % 2 else len(phase) - 1)
    if window_length > SAVGOL_POLYORDER:
        phase_smooth = savgol_filter(phase, window_length=window_length, polyorder=SAVGOL_POLYORDER)
phase_smooth = np.maximum.accumulate(phase_smooth)

lut_x_measured = phase_smooth / (2 * np.pi) * 65535
lut_y_measured = brightness
unique_indices = np.append(np.where(np.diff(lut_x_measured) > 0)[0], len(lut_x_measured) - 1)
lut_x_clean = lut_x_measured[unique_indices]
lut_y_clean = lut_y_measured[unique_indices]
if len(lut_x_clean) < 2:
    raise ValueError('Measured phase is not sufficiently increasing to build a LUT.')

interpolator = PchipInterpolator(lut_x_clean, lut_y_clean, extrapolate=False)
lut_input = np.arange(65536)
lut_output = interpolator(lut_input)
lut_output = np.where(
    np.isnan(lut_output),
    np.where(lut_input < lut_x_clean[0], lut_y_clean[0], lut_y_clean[-1]),
    lut_output,
)
lut_output = np.clip(np.round(lut_output), 0, 65535).astype(np.uint16)

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
with GENERATED_LUT_PATH.open('w', encoding='utf-8') as lut_file:
    for input_value, output_value in zip(lut_input, lut_output):
        lut_file.write(f'{input_value} {int(output_value)}\n')

print(f'Saved LUT to: {GENERATED_LUT_PATH}')
print(f'Measured phase stroke: {(phase.max() - phase.min()) / (2 * np.pi):.3f} x 2 * pi')

phase_figure = plt.figure(figsize=(8, 4))
plt.plot(brightness, phase, '.', alpha=0.4, label='Raw unwrapped phase')
plt.plot(brightness, phase_smooth, '-', label='Smoothed phase')
plt.xlabel('Brightness sent to SLM')
plt.ylabel('Measured phase [rad]')
plt.title('Chosen measured branch')
plt.legend()
plt.grid()
phase_figure.savefig(OUTPUT_PATH / 'chosen_phase_branch.png', dpi=150)
plt.show()

lut_figure = plt.figure(figsize=(8, 4))
plt.plot(lut_x_measured, lut_y_measured, '.', label='Measured inverse response')
plt.plot(lut_input, lut_output, '-', label='Interpolated LUT')
plt.xlabel('Input value = wanted phase, 0 to 65535')
plt.ylabel('Output brightness sent to SLM')
plt.title('Generated LUT')
plt.legend()
plt.grid()
lut_figure.savefig(OUTPUT_PATH / 'generated_lut.png', dpi=150)
plt.show()


In [ ]:
if not REFERENCE_LUT_PATH.is_file():
    raise FileNotFoundError(f'Comparison LUT not found: {REFERENCE_LUT_PATH}')

provided_lut = np.loadtxt(REFERENCE_LUT_PATH)
provided_input = provided_lut[:, 0]
provided_output = provided_lut[:, 1]

comparison_figure = plt.figure(figsize=(9, 5))
plt.plot(provided_input, provided_output, label='Provided 635 nm LUT', linewidth=2)
plt.plot(lut_input, lut_output, label='Generated LUT', linewidth=2)
plt.plot(lut_x_measured, lut_y_measured, '.', alpha=0.5, label='Measured points')
plt.xlabel('Input value = wanted phase, 0 to 65535')
plt.ylabel('Output brightness sent to SLM')
plt.title('Generated LUT vs provided 635 nm LUT')
plt.grid()
comparison_figure.savefig(OUTPUT_PATH / 'lut_comparison.png', dpi=150)
plt.legend()
plt.show()

provided_interpolated = np.interp(lut_input, provided_input, provided_output)
difference_figure = plt.figure(figsize=(9, 4))
plt.plot(lut_input, lut_output.astype(float) - provided_interpolated)
plt.axhline(0, linestyle='--')
plt.xlabel('Input value = wanted phase, 0 to 65535')
plt.ylabel('Generated LUT - provided LUT')
plt.title('Difference from provided 635 nm LUT')
plt.grid()
difference_figure.savefig(OUTPUT_PATH / 'lut_difference.png', dpi=150)
plt.show()
